# v0.40.0 — Model factories and the pytest plugin

## 1. Intro

Since v0.40.0 `surreal_orm_lite.testing` has the full SurrealDB-ORM's `ModelFactory` and `Faker`: a Factory Boy-style
class declares a model's test data, and generates instances from it.

- A declaration is a `Faker` provider (a new value for each instance), a static value (deep-copied for each instance)
  or a `SurrealFunc` (SurrealQL the server evaluates on `create()`).
- `build()` and `build_batch(n)` make unsaved instances, without a database; `create()` and `create_batch(n)` save
  them through the model's own `save()`.
- Mistakes fail early: a `Faker` checks its provider and keywords when it is built, a factory its names at the `class`
  statement and the overrides at the call.
- An opt-in pytest plugin, `surreal_orm_lite.testing.pytest_plugin`, gives each test a fresh in-memory database
  through its `surreal_db` fixture.

`Faker` values come from Python's `random` module: they are **test data, never secrets or tokens**. Factories behave
the same on SurrealDB 3.x, 2.x and the embedded engine, and the notebook runs top to bottom on both server lines. The
outputs below come from a SurrealDB 3.x server; a 2.x server prints the same lines.

## 2. Connect

`SURREALDB_HOST` and `SURREALDB_PORT` choose the server (default `localhost:8000`), over WebSocket.

In [1]:
import contextlib
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
# WebSocket (.../rpc), as the other notebooks.
SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 3. Three models

`NbUser` writes `NbPost`s; `NbWrote` is the edge between them (`in_` is the user, `out` the post). `token` and
`joined` will be generated by the server, so they have a default: `build()` leaves them out. The cell first removes
what an earlier run left, so the notebook can be re-executed in place.

In [2]:
from datetime import datetime

from surreal_orm_lite import BaseSurrealEdge, BaseSurrealModel, SurrealConfigDict


class NbUser(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="nb_user")

    id: str | None = None
    name: str
    email: str
    age: int
    role: str = "member"
    tags: list[str] = []
    token: str | None = None  # generated by the server on create()
    joined: datetime | None = None  # generated by the server on create()


class NbPost(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="nb_post")

    id: str | None = None
    title: str
    words: int = 0


class NbWrote(BaseSurrealEdge):
    model_config = SurrealConfigDict(table_name="nb_wrote")

    stars: int = 0


client = await SurrealDBConnectionManager.get_client()
for table in ("nb_wrote", "nb_post", "nb_user"):
    with contextlib.suppress(Exception):
        await client.query(f"REMOVE TABLE IF EXISTS {table};", {})
print("users:", await NbUser.objects().count())

users: 0


## 4. Factories

A factory names its model in `class Meta`; `ModelFactory[NbUser]` types `build()` and `create()` as `NbUser`. Each
public attribute is a declaration: `Faker("name")` draws a new name for each instance, `role = "member"` and `tags =
["new"]` are copied for each instance (two users never share a list), and the two `SurrealFunc` attributes are
evaluated by the server when `create()` saves the record. `NbWroteFactory` declares plain values and a `Faker` only:
an edge's `save()` cannot evaluate server expressions (§8).

In [3]:
from surreal_orm_lite import SurrealFunc
from surreal_orm_lite.testing import Faker, ModelFactory


class NbUserFactory(ModelFactory[NbUser]):
    class Meta:
        model = NbUser

    name = Faker("name")
    email = Faker("email")
    age = Faker("random_int", min=18, max=80)
    role = "member"
    tags = ["new"]
    token = SurrealFunc("<string> rand::uuid::v7()")
    joined = SurrealFunc("time::now()")


class NbPostFactory(ModelFactory[NbPost]):
    class Meta:
        model = NbPost

    title = Faker("sentence")
    words = Faker("random_int", min=100, max=900)


class NbWroteFactory(ModelFactory[NbWrote]):
    class Meta:
        model = NbWrote

    stars = Faker("random_int", min=1, max=5)


print(NbUserFactory.__name__, "builds", NbUserFactory.build().__class__.__name__)

NbUserFactory builds NbUser


## 5. `build()`: instances without a database

`build()` returns an unsaved instance: no id, no request, no signal, and the `SurrealFunc` fields keep their default
(`None`). `build_batch(3)` gives three instances with fresh values; an override replaces a declaration for the call.
Every provider draws from `random`, so after `random.seed(42)` the factory gives the same values again.

In [4]:
import random

random.seed(42)
user = NbUserFactory.build()
print(user.name, "|", user.email, "|", user.age, "|", user.role, user.tags)
print("id:", user.get_id(), "| token:", user.token, "| joined:", user.joined)

for member in NbUserFactory.build_batch(3):
    print(" ", member.name, member.age)

admins = NbUserFactory.build_batch(2, role="admin", tags=["staff"])
print("override:", [(admin.role, admin.tags) for admin in admins])
print("separate lists:", admins[0].tags is not admins[1].tags)

random.seed(42)
first = NbUserFactory.build()
random.seed(42)
print("same values after the same seed:", NbUserFactory.build().model_dump() == first.model_dump())

Uma Brown | alice.white227@mock.dev | 23 | member ['new']
id: None | token: None | joined: None
  Sam Wilson 30
  Wendy Lee 66
  Zoe Garcia 24
override: [('admin', ['staff']), ('admin', ['staff'])]
separate lists: True
same values after the same seed: True


## 6. `create()`: saved through `save()`

`create()` builds an instance and saves it with the model's `save()`, passing the `SurrealFunc` attributes as
`server_values`: SurrealDB generates `token` (a UUIDv7) and `joined` for each record, and the instance comes back with
its id and those values. `create_batch(5)` saves five records, one after the other.

In [5]:
random.seed(1)
alice = await NbUserFactory.create(name="Alice")
print("saved:", alice.get_id() is not None, "| name:", alice.name)
print("token is a server UUIDv7:", len(alice.token) == 36 and alice.token[14] == "7")
print("joined is a server datetime:", isinstance(alice.joined, datetime))

members = await NbUserFactory.create_batch(5)
print("users:", await NbUser.objects().count())
print("distinct tokens:", len({member.token for member in [alice, *members]}))
stored = await NbUser.objects().get(alice.get_id())
print("stored token equals the returned one:", stored.token == alice.token)

saved: True | name: Alice
token is a server UUIDv7: True
joined is a server datetime: True
users: 6
distinct tokens: 6
stored token equals the returned one: True


## 7. Edges

An edge factory takes its endpoints at the call, as `in_` and `out` — a saved instance or a `RecordID` —, and
`create()` saves the edge with its `INSERT RELATION`. Two authors write two posts each; the traversal from the first
author finds their two posts.

In [6]:
random.seed(7)
authors = await NbUserFactory.create_batch(2, role="author")
for author in authors:
    for _ in range(2):
        post = await NbPostFactory.create()
        wrote = await NbWroteFactory.create(in_=author, out=post)
        print(" ", author.name, "->", post.title, f"({wrote.stars} stars)")

posts = await authors[0].get_related(NbWrote, model_class=NbPost)
print(authors[0].name, "wrote:", sorted(post.title for post in posts))
print("edges:", await NbWrote.objects().count())

  Karen Jones -> Victor delta foxtrot quebec yankee. (5 stars)
  Karen Jones -> Bravo foxtrot oscar delta kilo oscar oscar oscar romeo charlie oscar. (3 stars)
  Sophia Taylor -> Oscar bravo bravo foxtrot romeo lima. (4 stars)
  Sophia Taylor -> Juliet golf echo uniform charlie hotel mike india lima papa bravo november. (3 stars)
Karen Jones wrote: ['Bravo foxtrot oscar delta kilo oscar oscar oscar romeo charlie oscar.', 'Victor delta foxtrot quebec yankee.']
edges: 4


## 8. Mistakes fail early

A model drops an unknown keyword silently, so the factory checks every name first: an unknown `Faker` provider when
the `Faker` is built, a misspelt declaration at the `class` statement, a misspelt override at the call. A `SurrealFunc`
on an edge factory is refused at the `class` statement, before any request.

In [7]:
try:
    Faker("nme")
except ValueError as error:
    print("ValueError:", error)

try:
    NbUserFactory.build(nmae="Bob")
except TypeError as error:
    print("TypeError:", error)

try:

    class TypoFactory(ModelFactory[NbUser]):
        class Meta:
            model = NbUser

        emial = Faker("email")

except TypeError as error:
    print("TypeError:", error)

try:

    class StarredFactory(ModelFactory[NbWrote]):
        class Meta:
            model = NbWrote

        stars = SurrealFunc("rand::int(1, 5)")

except TypeError as error:
    print("TypeError:", error)

ValueError: Unknown Faker provider: 'nme'
TypeError: build() got 'nmae', which is not a field of NbUser (fields: age, email, id, joined, name, role, tags, token)
TypeError: TypoFactory declares 'emial', which is not a field of NbUser (fields: age, email, id, joined, name, role, tags, token)
TypeError: StarredFactory declares SurrealFunc 'stars', but NbWrote is an edge: its save() (INSERT RELATION) cannot evaluate server values. Use DEFINE FIELD … DEFAULT on the edge table, or a plain value.


## 9. Fixtures from factories

`build()` gives a `SurrealFixture` its templates: `load()` saves a copy of each for the block and deletes them
afterwards.

In [8]:
from surreal_orm_lite.testing import SurrealFixture, fixture


@fixture
class Team(SurrealFixture):
    captain = NbUserFactory.build(id="captain", name="Ada", role="captain")
    rookie = NbUserFactory.build(id="rookie", name="Lin")


before = await NbUser.objects().count()
async with Team.load() as team:
    print("inside:", await NbUser.objects().count() - before, "more users;", team.captain.name, "is", team.captain.role)
print("after:", await NbUser.objects().count() - before, "more users")

inside: 2 more users; Ada is captain
after: 0 more users


## 10. The pytest plugin

Enable the plugin in a root `conftest.py` (`pytest_plugins = ["surreal_orm_lite.testing.pytest_plugin"]`) or with
`pytest -p surreal_orm_lite.testing.pytest_plugin`. Its `surreal_db` fixture gives each test a fresh, empty embedded
database (`mem://`), so a test run never touches a server.

This demo needs **pytest and pytest-asyncio in the kernel's environment** (`pip install pytest pytest-asyncio`): it
writes a tiny `conftest.py` and test module into a temporary directory and runs `pytest` on them in a subprocess. The
second test sees an empty database again.

In [9]:
import importlib.util
import re
import shutil
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

CONFTEST = """
pytest_plugins = ["surreal_orm_lite.testing.pytest_plugin"]
"""

TESTS = """
from surreal_orm_lite import BaseSurrealModel
from surreal_orm_lite.testing import Faker, ModelFactory


class Player(BaseSurrealModel):
    id: str | None = None
    name: str
    level: int = 1


class PlayerFactory(ModelFactory[Player]):
    class Meta:
        model = Player

    name = Faker("name")
    level = Faker("random_int", min=1, max=10)


async def test_create_batch(surreal_db):
    await PlayerFactory.create_batch(3)
    assert await Player.objects().count() == 3


async def test_each_test_starts_empty(surreal_db):
    assert await Player.objects().count() == 0
"""

if importlib.util.find_spec("pytest") is None or importlib.util.find_spec("pytest_asyncio") is None:
    print("skipped: install pytest and pytest-asyncio in this kernel to run the demo")
else:
    directory = tempfile.mkdtemp()
    try:
        Path(directory, "conftest.py").write_text(textwrap.dedent(CONFTEST))
        Path(directory, "test_players.py").write_text(textwrap.dedent(TESTS))
        result = subprocess.run(
            [
                sys.executable, "-m", "pytest", "-q",
                "-p", "no:cacheprovider", "-p", "no:randomly", "-p", "no:cov",
                "-o", "asyncio_mode=auto", "-o", "asyncio_default_fixture_loop_scope=function",
                directory,
            ],
            capture_output=True,
            text=True,
        )
        out = result.stdout
        passed = re.search(r"(\d+) passed", out)
        summary = f"inner pytest run: {passed.group(1) if passed else 0} passed"
        for word in ("failed", "errors?"):
            found = re.search(rf"(\d+) {word}", out)
            if found:
                summary += f", {found.group(0)}"
        print(summary)
        if result.returncode != 0:
            print(out)
            print(result.stderr)
    finally:
        shutil.rmtree(directory, ignore_errors=True)

inner pytest run: 2 passed


## 11. Cleanup

Remove the demo tables and close the connection.

In [10]:
for table in ("nb_wrote", "nb_post", "nb_user"):
    await client.query(f"REMOVE TABLE IF EXISTS {table};", {})
await SurrealDBConnectionManager.close_connection()
print("Done")

Done
